In [22]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [23]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [24]:
spark.version

'4.1.1'

In [25]:
import requests
import os

In [27]:
url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet'
local_filename = 'yellow_2025-11.parquet'

# Download the file
print(f"Downloading {url}...")
response = requests.get(url)
response.raise_for_status() 

In [28]:
with open(local_filename, "wb") as f:
    f.write(response.content)

print("File saved to:", os.path.abspath(local_filename))

File saved to: F:\Upskill 2026\DE-Zoomcamp-upSkill\6.Batch\yellow_2025-11.parquet


In [29]:
!ls -lh yellow_2025-11.parquet

-rw-r--r-- 1 Clarence Lee 197121 68M Mar  9 23:49 yellow_2025-11.parquet


In [30]:
df = spark.read.parquet('yellow_2025-11.parquet')

In [31]:
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

**Q3** Count records 

In [32]:
from pyspark.sql import functions as F

In [33]:
df.count()

4181444

**Q4** Longest trip

In [34]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [37]:
df.createOrReplaceTempView('yellow_2025_11')

In [38]:
spark.sql("""
SELECT
    MAX(trip_distance) AS longest_trip
FROM 
    yellow_2025_11
""").show()

+------------+
|longest_trip|
+------------+
|   256067.65|
+------------+



**Q6**

In [42]:
df = spark.read.csv("taxi_zone_lookup.csv", header=True)
df.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [43]:
df.createOrReplaceTempView('taxi_zone_lookup')

In [45]:
spark.sql("""
SELECT
    t.Zone AS pickup_zone,
    COUNT(1) AS pickup_zone_count
FROM 
    yellow_2025_11 AS y
LEFT JOIN 
    taxi_zone_lookup AS t
ON
    y.PULocationID = t.LocationID
GROUP BY
    t.Zone
ORDER BY
    pickup_zone_count
""").show()

+--------------------+-----------------+
|         pickup_zone|pickup_zone_count|
+--------------------+-----------------+
|Governor's Island...|                1|
|Eltingville/Annad...|                1|
|       Arden Heights|                1|
|       Port Richmond|                3|
|       Rikers Island|                4|
|   Rossville/Woodrow|                4|
| Green-Wood Cemetery|                4|
|         Great Kills|                4|
|         Jamaica Bay|                5|
|         Westerleigh|               12|
|        Crotona Park|               14|
|             Oakwood|               14|
|New Dorp/Midland ...|               14|
|       West Brighton|               14|
|       Willets Point|               15|
|Breezy Point/Fort...|               16|
|Saint George/New ...|               17|
|       Broad Channel|               18|
|     Mariners Harbor|               21|
|Heartland Village...|               22|
+--------------------+-----------------+
only showing top